In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score

In [2]:
file_path = 'Data/dane.csv'

heart_test = pd.read_csv('Data/heart_test.csv')
heart_train = pd.read_csv('Data/heart_train.csv')
diabetes_test = pd.read_csv('Data/diabetes_test.csv')
diabetes_train = pd.read_csv('Data/diabetes_train.csv')
cancer_test = pd.read_csv('Data/cancer_test.csv')
cancer_train = pd.read_csv('Data/cancer_train.csv')
alzheimer_test = pd.read_csv('Data/alzheimer_test.csv')
alzheimer_train = pd.read_csv('Data/alzheimer_train.csv')

datasets = {
    "heart": (heart_train, heart_test),
    "diabetes": (diabetes_train, diabetes_test),
    "cancer": (cancer_train, cancer_test),
    "alzheimer": (alzheimer_train, alzheimer_test)
}

# Uniform

In [5]:
n_random = 100
np.random.seed(42)

from scipy.stats import randint

param_dist_knn = {
    'n_neighbors': randint(1, 31),           # liczba sąsiadów w zakresie [1, 30]
    'weights': ['uniform', 'distance'],      # sposób ważenia sąsiadów
    'p': randint(1, 3),                      # 1 = Manhattan, 2 = Euklides
}



In [6]:
all_results = []

for name, (train, test) in datasets.items():
    print(f"Trenuję model KNN dla: {name}")

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = KNeighborsClassifier()

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist_knn,
        n_iter=100,
        scoring='roc_auc',
        cv=5,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    cv_results = pd.DataFrame(search.cv_results_)

    for i, params in enumerate(search.cv_results_['params']):
        tmp_model = KNeighborsClassifier(**params)
        tmp_model.fit(X_train, y_train)
        y_proba = tmp_model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)
    
        # podstawowy rekord
        result = {
            "dataset": name,
            "cv_roc_auc": search.cv_results_['mean_test_score'][i],
            "test_roc_auc": test_auc
        }
    
        # dodaj osobno każdy parametr jako kolumnę
        for param_name, param_value in params.items():
            result[param_name] = param_value
    
        all_results.append(result)


results_df = pd.DataFrame(all_results)

Trenuję model KNN dla: heart
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Trenuję model KNN dla: diabetes
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Trenuję model KNN dla: cancer
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Trenuję model KNN dla: alzheimer
Fitting 5 folds for each of 100 candidates, totalling 500 fits


In [7]:
best_per_dataset = (
    results_df
    .sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
)

param_cols = [col for col in results_df.columns if col not in ["dataset", "cv_roc_auc", "test_roc_auc"]]
params_df = best_per_dataset[param_cols]

aggregated_params = {}

for col in params_df.columns:
    if col.lower() in ["n_neighbors"]:
        # Średnia i zaokrąglenie do najbliższej liczby całkowitej
        aggregated_params[col] = int(round(params_df[col].mean()))
    else:
        aggregated_params[col] = params_df[col].mode().iloc[0]


mean_params = pd.Series(aggregated_params)

print("Średnie najlepsze parametry:")
print(mean_params)


Średnie najlepsze parametry:
n_neighbors          24
p                     1
weights        distance
dtype: object


In [8]:
mean_results = []

print(f"Testuję wspólne średnie parametry: {mean_params.to_dict()}")

for name, (train, test) in datasets.items():
    print(f"\nTrenuję model KNN na średnich parametrach dla: {name}")

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = KNeighborsClassifier(**mean_params.to_dict())
    model.fit(X_train, y_train)

    y_proba = model.predict_proba(X_test)[:, 1]
    mean_auc = roc_auc_score(y_test, y_proba)

    mean_results.append({
        "dataset": name,
        "star_test_roc_auc": mean_auc
    })

mean_df = pd.DataFrame(mean_results)

print("\nWyniki AUC dla wspólnych średnich parametrów na wszystkich zbiorach danych:")
print(mean_df)

Testuję wspólne średnie parametry: {'n_neighbors': 24, 'p': 1, 'weights': 'distance'}

Trenuję model KNN na średnich parametrach dla: heart

Trenuję model KNN na średnich parametrach dla: diabetes

Trenuję model KNN na średnich parametrach dla: cancer

Trenuję model KNN na średnich parametrach dla: alzheimer

Wyniki AUC dla wspólnych średnich parametrów na wszystkich zbiorach danych:
     dataset  star_test_roc_auc
0      heart           0.674140
1   diabetes           0.800597
2     cancer           0.772560
3  alzheimer           0.737553


In [9]:
results_df = results_df.merge(mean_df, on="dataset")
results_df["diff_from_star"] = results_df["star_test_roc_auc"] - results_df["test_roc_auc"]
results_df

results_df = results_df[
    ["dataset", "n_neighbors", "p", "weights", "cv_roc_auc", "test_roc_auc", "star_test_roc_auc", "diff_from_star"]
]

results_df.to_csv("Results/knn_random.csv", index=False)

In [10]:
results_df

,dataset,n_neighbors,p,weights,cv_roc_auc,test_roc_auc,star_test_roc_auc,diff_from_star
0,heart,7,2,uniform,0.637715,0.674644,0.674140,-0.000504
1,heart,15,1,distance,0.674662,0.670533,0.674140,0.003607
2,heart,29,1,uniform,0.676621,0.659206,0.674140,0.014933
3,heart,26,1,uniform,0.680318,0.665878,0.674140,0.008262
4,heart,11,1,distance,0.665385,0.672045,0.674140,0.002095
...,...,...,...,...,...,...,...,...
395,alzheimer,17,2,uniform,0.743065,0.708530,0.737553,0.029023
396,alzheimer,27,1,distance,0.768097,0.747429,0.737553,-0.009876
397,alzheimer,2,2,distance,0.630049,0.609513,0.737553,0.128040
398,alzheimer,23,1,distance,0.758384,0.738506,0.737553,-0.000953


In [31]:
# Creating a short summary dataset
results_df =  pd.read_csv("Results/knn_random.csv")

# Best parameters for each dataset
best_per_dataset = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
    .drop(['cv_roc_auc', 'diff_from_star'], axis=1)
)

In [8]:
# Deafault model

default_results = []
for name, (train, test) in datasets.items():
    
    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = KNeighborsClassifier()
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    score = roc_auc_score(y_test, y_proba)

    default_results.append({
        "dataset": name,
        "default_test_roc_auc": score
    })

default_df = pd.DataFrame(default_results)

In [ ]:
summary_df = best_per_dataset.merge(default_df, on="dataset")

In [33]:
# STAR row
mean_row = {
    "dataset": "STAR",
    **mean_params.to_dict(),
    # "ccp_alpha" :None, "max_depth": None, "min_samples_leaf": None, "min_samples_split": None,
    "test_roc_auc": None,
    "star_test_roc_auc": mean_df["star_test_roc_auc"].mean(),
    #"star_test_roc_auc": None,
    "default_test_roc_auc": None
}

summary_df = pd.concat([summary_df, pd.DataFrame([mean_row])], ignore_index=True)
summary_df


C:\Users\adawo\AppData\Local\Temp\ipykernel_2416\1105186936.py:12: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  summary_df = pd.concat([summary_df, pd.DataFrame([mean_row])], ignore_index=True)


,dataset,n_neighbors,p,weights,test_roc_auc,star_test_roc_auc,default_test_roc_auc
0,alzheimer,30,1,distance,0.753539,0.737553,0.676913
1,cancer,30,2,distance,0.780356,0.772560,0.735195
2,diabetes,27,1,uniform,0.812537,0.800597,0.712776
3,heart,10,2,distance,0.686630,0.674140,0.669854
4,STAR,24,1,distance,NaN,0.746212,NaN


In [34]:
summary_df

,dataset,n_neighbors,p,weights,test_roc_auc,star_test_roc_auc,default_test_roc_auc
0,alzheimer,30,1,distance,0.753539,0.737553,0.676913
1,cancer,30,2,distance,0.780356,0.772560,0.735195
2,diabetes,27,1,uniform,0.812537,0.800597,0.712776
3,heart,10,2,distance,0.686630,0.674140,0.669854
4,STAR,24,1,distance,NaN,0.746212,NaN


In [35]:
summary_df.to_csv("Results/knn_random_summary.csv", index=False)

# Bayesian

In [3]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer, Categorical
from sklearn.metrics import roc_auc_score

In [4]:
search_spaces = {
    'n_neighbors': Integer(1, 30),                  # zakres [1, 30]
    'weights': Categorical(['uniform', 'distance']), # wybór spośród dwóch opcji
    'p': Integer(1, 2),                             # 1 = Manhattan, 2 = Euklides
}

In [5]:
from sklearn.neighbors import KNeighborsClassifier
from skopt import BayesSearchCV
from sklearn.metrics import roc_auc_score
import pandas as pd

all_results = []

for name, (train, test) in datasets.items():
    print(f"Trenuję model KNN dla: {name}")

    X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
    X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

    model = KNeighborsClassifier()

    search = BayesSearchCV(
        estimator=model,
        search_spaces=search_spaces,
        n_iter=100,
        scoring='roc_auc',
        cv=5,
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    search.fit(X_train, y_train)

    cv_results = pd.DataFrame(search.cv_results_)

    for i, params in enumerate(search.cv_results_['params']):
        tmp_model = KNeighborsClassifier(**params)
        tmp_model.fit(X_train, y_train)
        y_proba = tmp_model.predict_proba(X_test)[:, 1]
        test_auc = roc_auc_score(y_test, y_proba)

        result = {
            "dataset": name,
            "cv_roc_auc": search.cv_results_['mean_test_score'][i],
            "test_roc_auc": test_auc
        }

        result.update(params)

        all_results.append(result)

results_df = pd.DataFrame(all_results)


Trenuję model KNN dla: heart
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of

C:\Users\adawo\anaconda3\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [25, 1, 'distance'] before, using random point [7, 1, 'uniform']
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits


C:\Users\adawo\anaconda3\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [28, 1, 'distance'] before, using random point [24, 1, 'distance']
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fi

...

Fitting 5 folds for each of 1 candidates, totalling 5 fits


C:\Users\adawo\anaconda3\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [29, 1, 'distance'] before, using random point [9, 2, 'uniform']
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits


C:\Users\adawo\anaconda3\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point [29, 1, 'distance'] before, using random point [7, 2, 'distance']
  warnings.warn(


Fitting 5 folds for each of 1 candidates, totalling 5 fits
Fitting 5 folds for each of 1 candidates, totalling 5 fits


In [42]:
# best_per_dataset = (
#     results_df
#     .sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False])
#     .groupby("dataset", as_index=False)
#     .first()
# )

# param_cols = [col for col in results_df.columns if col not in ["dataset", "cv_roc_auc", "test_roc_auc"]]
# params_df = best_per_dataset[param_cols]

# aggregated_params = {}

# for col in params_df.columns:
#     if col.lower() in ["n_neighbors"]:
#         # Średnia i zaokrąglenie do najbliższej liczby całkowitej
#         aggregated_params[col] = int(round(params_df[col].mean()))
#     else:
#         aggregated_params[col] = params_df[col].mode().iloc[0]


# mean_params = pd.Series(aggregated_params)

# print("Średnie najlepsze parametry:")
# print(mean_params)

Średnie najlepsze parametry:
n_neighbors          26
p                     1
weights        distance
dtype: object


In [43]:
# mean_results = []

# print(f"Testuję wspólne średnie parametry: {mean_params.to_dict()}")

# for name, (train, test) in datasets.items():
#     print(f"\nTrenuję model KNN na średnich parametrach dla: {name}")

#     X_train, y_train = train.iloc[:, :-1], train.iloc[:, -1]
#     X_test, y_test = test.iloc[:, :-1], test.iloc[:, -1]

#     model = KNeighborsClassifier(**mean_params.to_dict())
#     model.fit(X_train, y_train)

#     y_proba = model.predict_proba(X_test)[:, 1]
#     mean_auc = roc_auc_score(y_test, y_proba)

#     mean_results.append({
#         "dataset": name,
#         "star_test_roc_auc": mean_auc
#     })

# mean_df = pd.DataFrame(mean_results)

# print("\nWyniki AUC dla wspólnych średnich parametrów na wszystkich zbiorach danych:")
# print(mean_df)

Testuję wspólne średnie parametry: {'n_neighbors': 26, 'p': 1, 'weights': 'distance'}

Trenuję model KNN na średnich parametrach dla: heart

Trenuję model KNN na średnich parametrach dla: diabetes

Trenuję model KNN na średnich parametrach dla: cancer

Trenuję model KNN na średnich parametrach dla: alzheimer

Wyniki AUC dla wspólnych średnich parametrów na wszystkich zbiorach danych:
     dataset  star_test_roc_auc
0      heart           0.671580
1   diabetes           0.804060
2     cancer           0.772064
3  alzheimer           0.743935


In [6]:
# results_df = results_df.merge(mean_df, on="dataset")
# results_df["diff_from_mean"] = results_df["star_test_roc_auc"] - results_df["test_roc_auc"]
# results_df

results_df = results_df[
    ["dataset", "n_neighbors", "p", "weights", "cv_roc_auc", "test_roc_auc"]
]

results_df.to_csv("Results/knn_bayes.csv", index=False)

In [7]:
results_df

,dataset,n_neighbors,p,weights,cv_roc_auc,test_roc_auc
0,heart,13,2,distance,0.668994,0.677476
1,heart,25,2,uniform,0.673216,0.670940
2,heart,14,2,uniform,0.671440,0.669621
3,heart,25,1,distance,0.688741,0.676002
4,heart,24,1,distance,0.690304,0.674140
...,...,...,...,...,...,...
395,alzheimer,13,2,uniform,0.736140,0.706768
396,alzheimer,28,2,uniform,0.759414,0.734634
397,alzheimer,9,2,uniform,0.712499,0.670145
398,alzheimer,7,2,distance,0.709415,0.674592


In [9]:
best_per_dataset_2 = (
    results_df.sort_values(by=["dataset", "test_roc_auc"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
    .drop(['cv_roc_auc'], axis=1)
)

summary_2_df = best_per_dataset_2.merge(default_df, on="dataset")

In [10]:
summary_2_df

,dataset,n_neighbors,p,weights,test_roc_auc,default_test_roc_auc
0,alzheimer,30,1,distance,0.753539,0.676913
1,cancer,30,2,distance,0.780356,0.735195
2,diabetes,27,1,uniform,0.812537,0.712776
3,heart,19,1,distance,0.684574,0.669854


In [11]:
summary_2_df.to_csv("Results/knn_bayes_summary.csv", index=False)